In [1]:
#!/usr/bin/env python3
import math
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm

# ---- set these ----
RUN_DIR = Path("/lambda/nfs/neel/Research/runs/dinov2/imagenet1k_sae/layer6_high_norm_tokens")
SHARDS_DIR = RUN_DIR / "shards"
LIMIT_SHARDS = 0          # 0 = all
BATCH = 200_000
SAMPLE_SIZE = 2_000_000   # reservoir sample for quantiles

TT_REG = 0  # patch tokens are token_type != 0

def reservoir_update(sample, filled, seen, new_vals):
    # sample: np array size SAMPLE_SIZE
    # filled: how many slots filled so far
    # seen: how many total values seen so far
    for v in new_vals:
        seen += 1
        if filled < sample.size:
            sample[filled] = v
            filled += 1
        else:
            j = np.random.randint(1, seen + 1)
            if j <= sample.size:
                sample[j - 1] = v
    return filled, seen

def main():
    paths = sorted(SHARDS_DIR.glob("shard_*.pt"))
    if not paths:
        raise FileNotFoundError(f"No shards found in {SHARDS_DIR}")
    if LIMIT_SHARDS > 0:
        paths = paths[:LIMIT_SHARDS]

    # Welford streaming mean/std
    n = 0
    mean = 0.0
    M2 = 0.0

    # extra stats
    maxv = -1.0
    gt150 = 0

    # reservoir sample for quantiles
    sample = np.empty(SAMPLE_SIZE, dtype=np.float32)
    filled = 0
    seen = 0

    for p in tqdm(paths, desc="scan shards", dynamic_ncols=True):
        shard = torch.load(p, map_location="cpu")
        X = shard["vecs"].to(torch.float32)          # (N,D)
        tt = shard["token_type"].to(torch.int64)     # (N,)

        mask = (tt != TT_REG)
        if mask.sum().item() == 0:
            continue

        Xp = X[mask]  # patch tokens only
        total = Xp.shape[0]

        for s in range(0, total, BATCH):
            xb = Xp[s:s+BATCH]
            norms = torch.linalg.vector_norm(xb, dim=1).numpy()  # float32

            # update max and gt150
            maxv = max(maxv, float(norms.max(initial=0.0)))
            gt150 += int((norms > 150.0).sum())

            # Welford update
            for v in norms:
                n += 1
                delta = float(v) - mean
                mean += delta / n
                M2 += delta * (float(v) - mean)

            # reservoir
            filled, seen = reservoir_update(sample, filled, seen, norms)

    var = (M2 / (n - 1)) if n > 1 else 0.0
    std = math.sqrt(var)

    used = sample[:filled]
    qs = np.quantile(used, [0.95, 0.975, 0.99, 0.995]) if filled > 0 else [float("nan")]*4

    print("\nPATCH TOKEN NORM STATS")
    print(f"count: {n}")
    print(f"mean:  {mean:.6f}")
    print(f"std:   {std:.6f}")
    print(f"max:   {maxv:.6f}")
    print(f"q95:   {qs[0]:.6f}")
    print(f"q97.5: {qs[1]:.6f}")
    print(f"q99:   {qs[2]:.6f}")
    print(f"q99.5: {qs[3]:.6f}")
    print(f"% > 150: {100.0 * gt150 / max(1, n):.4f}%")

if __name__ == "__main__":
    main()


scan shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 635/635 [04:30<00:00,  2.35it/s]


PATCH TOKEN NORM STATS
count: 10249336
mean:  6.311708
std:   0.799097
max:   17.150150
q95:   7.493863
q97.5: 7.719136
q99:   8.030545
q99.5: 8.278584
% > 150: 0.0000%
